###### Imports and Settings

In [1]:
import pandas as pd
import numpy as np
import requests
#import io
import pickle
from collections import deque
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 150)
pd.options.mode.chained_assignment = None  # default='warn'

In [2]:
def percent(x, y):
    return x/y

## Calculations

In [3]:
data = pd.read_csv('../../data/SF12000.csv')

## Population + Projections Summary: 

In [4]:
data['Population'] = data['agebysex_total_series']
data = data.drop(columns = ['agebysex_total_series','pop'])

## Age Sex Race Summary:

### Age Brackets
+ M and F U5, 5 to 9, 10 to 14, 15 to 17, 18 to 24, 25 to 34, 35 to 44, 45 to 54, 55 to 64, 65 to 74, 75 to 84, 85+  
+ age brackets: under 18, 18 to 54, 55+  
+ 65+ 

In [5]:
#small groups m and f
data['Male Under 5'] = data['age_m_u5']
data['Female Under 5'] = data['age_f_u5']
data['Male 5 to 9'] = data['age_m_5to9']
data['Female 5 to 9'] = data['age_f_5to9']
data['Male 10 to 14'] = data['age_m_10to14']
data['Female 10 to 14'] = data['age_f_10to14']
data['Male 15 to 17'] = data['age_m_15to17']
data['Female 15 to 17'] = data['age_f_15to17']
data['Male 18 to 24'] = data['age_m_18to19']+data['age_m_20']+data['age_m_21']+data['age_m_22to24']
data['Female 18 to 24'] = data['age_f_18to19']+data['age_f_20']+data['age_f_21']+data['age_f_22to24']
data['Male 25 to 34'] = data['age_m_25to29']+data['age_m_30to34']
data['Female 25 to 34'] = data['age_f_25to29']+data['age_f_30to34']
data['Male 35 to 44'] = data['age_m_35to39']+data['age_m_40to44']
data['Female 35 to 44'] = data['age_f_35to39']+data['age_f_40to44']
data['Male 45 to 54'] = data['age_m_45to49']+data['age_m_50to54']
data['Female 45 to 54'] = data['age_f_45to49']+data['age_f_50to54']
data['Male 55 to 64'] = data['age_m_55to59']+data['age_m_60to61']+data['age_m_62to64']
data['Female 55 to 64'] = data['age_f_55to59']+data['age_f_60to61']+data['age_f_62to64']
data['Male 65 to 74'] = data['age_m_65to66']+data['age_m_67to69']+data['age_m_70to74']
data['Female 65 to 74'] = data['age_f_65to66']+data['age_f_67to69']+data['age_f_70to74']
data['Male 75 to 84'] = data['age_m_75to79']+data['age_m_80to84']
data['Female 75 to 84'] = data['age_f_75to79']+data['age_f_80to84']
data['Male 85 and Older'] = data['age_m_85+']
data['Female 85 and Older'] = data['age_f_85+']

#age brackets
u18list = [data['Male Under 5'],data['Female Under 5'],data['Male 5 to 9'],data['Female 5 to 9'],data['Male 10 to 14'],data['Female 10 to 14'],data['Male 15 to 17'],
           data['Female 15 to 17']]
data['Age:Under 18'] = sum(u18list)
eighteento54list = [data['Male 18 to 24'],data['Female 18 to 24'],data['Male 25 to 34'],data['Female 25 to 34'],data['Male 35 to 44'],data['Female 35 to 44'],
              data['Male 45 to 54'],data['Female 45 to 54']]
data['Age:18 to 54'] = sum(eighteento54list)
fifty5pluslist = [data['Male 55 to 64'],data['Female 55 to 64'],data['Male 65 to 74'],data['Female 65 to 74'],data['Male 75 to 84'],data['Female 75 to 84'],
                  data['Male 85 and Older'],data['Female 85 and Older']]
data['Age:55 and Older'] = sum(fifty5pluslist)

#65+
sixty5pluslist = [data['Male 65 to 74'],data['Female 65 to 74'],data['Male 75 to 84'],data['Female 75 to 84'],data['Male 85 and Older'],data['Female 85 and Older']]
data['Age:65 and Older'] = sum(sixty5pluslist)

cols = ['age_total_male','age_total_female','age_m_u5','age_m_5to9','age_m_10to14','age_m_15to17','age_m_18to19','age_m_20','age_m_21','age_m_22to24','age_m_25to29','age_m_30to34','age_m_35to39',
        'age_m_40to44','age_m_45to49','age_m_50to54','age_m_55to59','age_m_60to61','age_m_62to64','age_m_65to66','age_m_67to69','age_m_70to74','age_m_75to79',
        'age_m_80to84','age_m_85+','age_f_u5','age_f_5to9','age_f_10to14','age_f_15to17','age_f_18to19','age_f_20','age_f_21','age_f_22to24','age_f_25to29',
        'age_f_30to34','age_f_35to39','age_f_40to44','age_f_45to49','age_f_50to54','age_f_55to59','age_f_60to61','age_f_62to64','age_f_65to66','age_f_67to69',
        'age_f_70to74','age_f_75to79','age_f_80to84','age_f_85+']
data = data.drop(columns = cols)

### Race/Ethnicity #s   
+ White Alone  
+ Black/African American Alone  
+ American Indian Alaska Native Alone  
+ Asian Alone  
+ Native Hawaiian/Other Pacific Islander Alone  
+ Some Other Race Alone  
+ Two or More Races  
+ Hispanic/Latino  
+ White, Not Hispanic/Latino  
+ Total Minority (non-white, non Hispanic/Latino)

In [6]:
#White Alone
data['White Alone'] = data['raceeth_white_alone']
data['White Alone %'] = data['White Alone']/data['Population']
#Black or African American Alone 
data['Black or African American Alone'] = data['raceeth_blackafricanamerican_alone']
data['Black or African American Alone %'] = data['Black or African American Alone']/data['Population']
#American Indian and Alaska Native Alone
data['American Indian Alaska Native Alone'] = data['raceeth_americanindianalaskanative_alone']
data['American Indian Alaska Native Alone %'] = data['American Indian Alaska Native Alone']/data['Population']
#Asian Alone
data['Asian Alone'] = data['raceeth_asian_alone']
data['Asian Alone %'] = data['Asian Alone']/data['Population']
#Native Hawaiian Other Pacific Islander Alone
data['Native Hawaiian Other Pacific Islander Alone'] = data['raceeth_nativehawaiianotherpacificislander_alone']
data['Native Hawaiian Other Pacific Islander Alone %'] = data['Native Hawaiian Other Pacific Islander Alone']/data['Population']
#Some Other Race Alone
data['Some Other Race Alone'] = data['raceeth_someotherrace_alone']
data['Some Other Race Alone %'] = data['Some Other Race Alone']/data['Population']
#Two or More Races
data['Two or More Races'] = data['raceeth_twoormoreraces_alone']
data['Two or More Races %'] = data['Two or More Races']/data['Population']
#Hispanic or Latino
data['Hispanic or Latino'] = data['raceeth_hispanicorlatino']
data['Hispanic or Latino %'] = data['Hispanic or Latino']/data['Population']
#Data Minority
data['Minority'] = data['Population'] - data['raceeth_whitealone_nothispanicorlatino']
data['Minority %'] = data['Minority']/data['Population']
cols = ['raceeth_total_series','raceeth_white_alone','raceeth_blackafricanamerican_alone','raceeth_americanindianalaskanative_alone','raceeth_asian_alone',
        'raceeth_nativehawaiianotherpacificislander_alone','raceeth_someotherrace_alone','raceeth_twoormoreraces_alone','raceeth_hispanicorlatino',
        'raceeth_whitealone_nothispanicorlatino','raceeth_total_oneracealone']
data = data.drop(columns = cols)

## Household Summary:  

### Households by Household Type  
+ Total Households  
+ Family Households  
+ Family Households: Married-Couple Family  
+ Family Households: Other Family  
+ Family Households: Other Family: Male Householder no Wife Present  
+ Family Households: Other Family: Female Householder no Husband Present  
+ Nonfamily Households  
+ Nonfamily Household: Male Householder  
+ Nonfamily Household: Female Householder  

*They don't have the nonfamily male or female - replaced by Householder alone or not alone*

In [7]:
data['Total Households'] = data['hhtype_total_series']
data['Family Households'] = data['hhtype_twoormore_family']
data['Family Households: Married Couple Family'] = data['hhtype_twoormore_family_marriedcouplefam']
data['Family Households: Not Married Couple Family'] = data['hhtype_twoormore_family_otherfam']
data['Family Households: Not Married Couple: Male no Spouse'] = data['hhtype_twoormore_family_otherfam_malenospouse']
data['Family Households: Not Married Couple: Female no Spouse'] = data['hhtype_twoormore_family_otherfam_femalenospouse']
data['Nonfamily Households'] = data['hhtype_twoormore_nonfamily'] + data['hhtype_oneperson']
data['Nonfamily Households: Householder Alone'] = data['hhtype_oneperson']
data['Nonfamily Households: Householder not Alone'] = data['hhtype_twoormore_nonfamily']

cols = ['hhtype_total_series','hhtype_oneperson','hhtype_oneperson_male','hhtype_oneperson_female','hhtype_twoormore','hhtype_twoormore_family',
        'hhtype_twoormore_family_marriedcouplefam','hhtype_twoormore_family_marriedcouplefam_ownchildrenunder18',
        'hhtype_twoormore_family_marriedcouplefam_noownchildrenunder18','hhtype_twoormore_family_otherfam','hhtype_twoormore_family_otherfam_malenospouse',
        'hhtype_twoormore_family_otherfam_malenospouse_ownchildrenunder18','hhtype_twoormore_family_otherfam_malenospouse_noownchildrenunder18',
        'hhtype_twoormore_family_otherfam_femalenospouse','hhtype_twoormore_family_otherfam_femalenospouse_ownchildrenunder18',
        'hhtype_twoormore_family_otherfam_femalenospouse_noownchildrenunder18','hhtype_twoormore_nonfamily','hhtype_twoormore_nonfamily_male',
        'hhtype_twoormore_nonfamily_female']
data = data.drop(columns = cols)

### Average Household Size 
Median - drop for incorporated and unincorporated

In [8]:
data['Average Household Size'] = data['hhsize_avg']
data = data.drop(columns = ['hhsize_avg'])

### Occupancy Status, and Percentages  
+ Occupancy Total Households
+ Occupied  
+ Vacant  

In [9]:
data['Occupancy:Total Households'] = data['occupancy_total_series']
data['Occupancy:Occupied Units'] = data['occupancy_occupiedunits']
data['Occupancy%:Occupied Units'] = percent(data['Occupancy:Occupied Units'], data['Occupancy:Total Households'])
data['Occupancy:Vacant Units'] = data['occupancy_vacantunits']
data['Occupancy%:Vacant Units'] = percent(data['Occupancy:Vacant Units'], data['Occupancy:Total Households'])

cols = ['occupancy_total_series','occupancy_occupiedunits','occupancy_vacantunits']
data = data.drop(columns = cols)

### Tenure, and Percentages  
+ Tenure Total Households  
+ Owners  
+ Renters

In [10]:
data['Tenure:Total Households'] = data['tenure_total_series']
data['Tenure:Owners'] = data['tenure_owneroccunits']
data['Tenure%:Owners'] = percent(data['Tenure:Owners'], data['Tenure:Total Households'])
data['Tenure:Renters'] = data['tenure_renteroccunits']
data['Tenure%:Renters'] = percent(data['Tenure:Renters'], data['Tenure:Total Households'])
cols = ['units_allhousing','tenure_total_series','tenure_owneroccunits','tenure_renteroccunits']
data = data.drop(columns = cols)

In [11]:
data = data.drop(columns = ['StateFIPS', 'GeoFIPS'])

In [12]:
data.head()

,NAME,Population,Male Under 5,Female Under 5,Male 5 to 9,Female 5 to 9,Male 10 to 14,Female 10 to 14,Male 15 to 17,Female 15 to 17,Male 18 to 24,Female 18 to 24,Male 25 to 34,Female 25 to 34,Male 35 to 44,Female 35 to 44,Male 45 to 54,Female 45 to 54,Male 55 to 64,Female 55 to 64,Male 65 to 74,Female 65 to 74,Male 75 to 84,Female 75 to 84,Male 85 and Older,Female 85 and Older,Age:Under 18,Age:18 to 54,Age:55 and Older,Age:65 and Older,White Alone,White Alone %,Black or African American Alone,Black or African American Alone %,American Indian Alaska Native Alone,American Indian Alaska Native Alone %,Asian Alone,Asian Alone %,Native Hawaiian Other Pacific Islander Alone,Native Hawaiian Other Pacific Islander Alone %,Some Other Race Alone,Some Other Race Alone %,Two or More Races,Two or More Races %,Hispanic or Latino,Hispanic or Latino %,Minority,Minority %,Total Households,Family Households,Family Households: Married Couple Family,Family Households: Not Married Couple Family,Family Households: Not Married Couple: Male no Spouse,Family Households: Not Married Couple: Female no Spouse,Nonfamily Households,Nonfamily Households: Householder Alone,Nonfamily Households: Householder not Alone,Average Household Size,Occupancy:Total Households,Occupancy:Occupied Units,Occupancy%:Occupied Units,Occupancy:Vacant Units,Occupancy%:Vacant Units,Tenure:Total Households,Tenure:Owners,Tenure%:Owners,Tenure:Renters,Tenure%:Renters
0,"Cheatham County, Tennessee",35912.0,1372.0,1187.0,1473.0,1330.0,1487.0,1398.0,829.0,854.0,1367.0,1242.0,2521.0,2688.0,3404.0,3429.0,2642.0,2533.0,1559.0,1512.0,878.0,962.0,361.0,531.0,88.0,265.0,9930.0,19826.0,6156.0,3085.0,34783.0,0.968562,532.0,0.014814,135.0,0.003759,63.0,0.001754,17.0,0.000473,130.0,0.003620,252.0,0.007017,437.0,0.012169,1406.0,0.039151,12878.0,10162.0,8356.0,1806.0,564.0,1242.0,2716.0,2177.0,539.0,2.76,13508.0,12878.0,0.953361,630.0,0.046639,12878.0,10773.0,0.836543,2105.0,0.163457
1,"Davidson County, Tennessee",569891.0,19466.0,18347.0,18200.0,17524.0,17089.0,16143.0,10120.0,9558.0,32612.0,33586.0,51053.0,49134.0,46237.0,47262.0,36358.0,38676.0,20839.0,24243.0,13996.0,19402.0,7902.0,14142.0,1993.0,6009.0,126447.0,334918.0,108526.0,63444.0,381783.0,0.669923,147696.0,0.259165,1679.0,0.002946,13275.0,0.023294,403.0,0.000707,13816.0,0.024243,11239.0,0.019721,26091.0,0.045782,198741.0,0.348735,237405.0,138106.0,94784.0,43322.0,9283.0,34039.0,99299.0,79249.0,20050.0,2.30,252977.0,237405.0,0.938445,15572.0,0.061555,237405.0,131340.0,0.553232,106065.0,0.446768
2,"Dickson County, Tennessee",43156.0,1538.0,1437.0,1686.0,1584.0,1726.0,1601.0,957.0,957.0,1744.0,1749.0,3010.0,3108.0,3535.0,3581.0,2828.0,2944.0,2036.0,2066.0,1304.0,1520.0,656.0,1041.0,138.0,410.0,11486.0,22499.0,9171.0,5069.0,40243.0,0.932501,1978.0,0.045834,172.0,0.003986,116.0,0.002688,5.0,0.000116,204.0,0.004727,438.0,0.010149,484.0,0.011215,3136.0,0.072667,16473.0,12175.0,9604.0,2571.0,670.0,1901.0,4298.0,3678.0,620.0,2.59,17614.0,16473.0,0.935222,1141.0,0.064778,16473.0,12539.0,0.761185,3934.0,0.238815
3,"Houston County, Tennessee",8088.0,268.0,273.0,311.0,243.0,266.0,298.0,174.0,137.0,307.0,285.0,520.0,484.0,555.0,548.0,546.0,567.0,467.0,489.0,343.0,384.0,189.0,264.0,53.0,117.0,1970.0,3812.0,2306.0,1350.0,7650.0,0.945846,268.0,0.033136,15.0,0.001855,10.0,0.001236,5.0,0.000618,63.0,0.007789,77.0,0.009520,101.0,0.012488,477.0,0.058976,3216.0,2300.0,1833.0,467.0,134.0,333.0,916.0,815.0,101.0,2.46,3901.0,3216.0,0.824404,685.0,0.175596,3216.0,2476.0,0.769900,740.0,0.230100
4,"Humphreys County, Tennessee",17929.0,567.0,505.0,610.0,597.0,660.0,584.0,402.0,357.0,712.0,654.0,1073.0,1173.0,1352.0,1337.0,1259.0,1317.0,1066.0,1049.0,705.0,823.0,339.0,526.0,74.0,188.0,4282.0,8877.0,4770.0,2655.0,17125.0,0.955156,527.0,0.029394,48.0,0.002677,46.0,0.002566,2.0,0.000112,29.0,0.001617,152.0,0.008478,148.0,0.008255,899.0,0.050142,7238.0,5145.0,4144.0,1001.0,266.0,735.0,2093.0,1808.0,285.0,2.44,8482.0,7238.0,0.853336,1244.0,0.146664,7238.0,5641.0,0.77935

In [13]:
data = data.set_index('NAME')
data = data.add_suffix(' 2000')
data = data.reset_index()

In [14]:
data.head()

,NAME,Population 2000,Male Under 5 2000,Female Under 5 2000,Male 5 to 9 2000,Female 5 to 9 2000,Male 10 to 14 2000,Female 10 to 14 2000,Male 15 to 17 2000,Female 15 to 17 2000,Male 18 to 24 2000,Female 18 to 24 2000,Male 25 to 34 2000,Female 25 to 34 2000,Male 35 to 44 2000,Female 35 to 44 2000,Male 45 to 54 2000,Female 45 to 54 2000,Male 55 to 64 2000,Female 55 to 64 2000,Male 65 to 74 2000,Female 65 to 74 2000,Male 75 to 84 2000,Female 75 to 84 2000,Male 85 and Older 2000,Female 85 and Older 2000,Age:Under 18 2000,Age:18 to 54 2000,Age:55 and Older 2000,Age:65 and Older 2000,White Alone 2000,White Alone % 2000,Black or African American Alone 2000,Black or African American Alone % 2000,American Indian Alaska Native Alone 2000,American Indian Alaska Native Alone % 2000,Asian Alone 2000,Asian Alone % 2000,Native Hawaiian Other Pacific Islander Alone 2000,Native Hawaiian Other Pacific Islander Alone % 2000,Some Other Race Alone 2000,Some Other Race Alone % 2000,Two or More Races 2000,Two or More Races % 2000,Hispanic or Latino 2000,Hispanic or Latino % 2000,Minority 2000,Minority % 2000,Total Households 2000,Family Households 2000,Family Households: Married Couple Family 2000,Family Households: Not Married Couple Family 2000,Family Households: Not Married Couple: Male no Spouse 2000,Family Households: Not Married Couple: Female no Spouse 2000,Nonfamily Households 2000,Nonfamily Households: Householder Alone 2000,Nonfamily Households: Householder not Alone 2000,Average Household Size 2000,Occupancy:Total Households 2000,Occupancy:Occupied Units 2000,Occupancy%:Occupied Units 2000,Occupancy:Vacant Units 2000,Occupancy%:Vacant Units 2000,Tenure:Total Households 2000,Tenure:Owners 2000,Tenure%:Owners 2000,Tenure:Renters 2000,Tenure%:Renters 2000
0,"Cheatham County, Tennessee",35912.0,1372.0,1187.0,1473.0,1330.0,1487.0,1398.0,829.0,854.0,1367.0,1242.0,2521.0,2688.0,3404.0,3429.0,2642.0,2533.0,1559.0,1512.0,878.0,962.0,361.0,531.0,88.0,265.0,9930.0,19826.0,6156.0,3085.0,34783.0,0.968562,532.0,0.014814,135.0,0.003759,63.0,0.001754,17.0,0.000473,130.0,0.003620,252.0,0.007017,437.0,0.012169,1406.0,0.039151,12878.0,10162.0,8356.0,1806.0,564.0,1242.0,2716.0,2177.0,539.0,2.76,13508.0,12878.0,0.953361,630.0,0.046639,12878.0,10773.0,0.836543,2105.0,0.163457
1,"Davidson County, Tennessee",569891.0,19466.0,18347.0,18200.0,17524.0,17089.0,16143.0,10120.0,9558.0,32612.0,33586.0,51053.0,49134.0,46237.0,47262.0,36358.0,38676.0,20839.0,24243.0,13996.0,19402.0,7902.0,14142.0,1993.0,6009.0,126447.0,334918.0,108526.0,63444.0,381783.0,0.669923,147696.0,0.259165,1679.0,0.002946,13275.0,0.023294,403.0,0.000707,13816.0,0.024243,11239.0,0.019721,26091.0,0.045782,198741.0,0.348735,237405.0,138106.0,94784.0,43322.0,9283.0,34039.0,99299.0,79249.0,20050.0,2.30,252977.0,237405.0,0.938445,15572.0,0.061555,237405.0,131340.0,0.553232,106065.0,0.446768
2,"Dickson County, Tennessee",43156.0,1538.0,1437.0,1686.0,1584.0,1726.0,1601.0,957.0,957.0,1744.0,1749.0,3010.0,3108.0,3535.0,3581.0,2828.0,2944.0,2036.0,2066.0,1304.0,1520.0,656.0,1041.0,138.0,410.0,11486.0,22499.0,9171.0,5069.0,40243.0,0.932501,1978.0,0.045834,172.0,0.003986,116.0,0.002688,5.0,0.000116,204.0,0.004727,438.0,0.010149,484.0,0.011215,3136.0,0.072667,16473.0,12175.0,9604.0,2571.0,670.0,1901.0,4298.0,3678.0,620.0,2.59,17614.0,16473.0,0.935222,1141.0,0.064778,16473.0,12539.0,0.761185,3934.0,0.238815
3,"Houston County, Tennessee",8088.0,268.0,273.0,311.0,243.0,266.0,298.0,174.0,137.0,307.0,285.0,520.0,484.0,555.0,548.0,546.0,567.0,467.0,489.0,343.0,384.0,189.0,264.0,53.0,117.0,1970.0,3812.0,2306.0,1350.0,7650.0,0.945846,268.0,0.033136,15.0,0.001855,10.0,0.001236,5.0,0.000618,63.0,0.007789,77.0,0.009520,101.0,0.012488,477.0,0.058976,3216.0,2300.0,1833.0,467.0,134.0,333.0,916.0,815.0,101.0,2.46,3901.0,3216.0,0.824404,685.0,0.175596,3216.0,2476.0,0.769900,740.0,0.230100
4,"Humphreys County, Tennessee",17929.0,567.0,505.0,610.0,597.0,660.0,584.0,402.0,357.0,712.0,654.0,1073.0,1173.0,1352.0,1337.0,1259.0,1317.0

In [15]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 81 entries, 0 to 80
Data columns (total 68 columns):
 #   Column                                                        Non-Null Count  Dtype  
---  ------                                                        --------------  -----  
 0   NAME                                                          81 non-null     object 
 1   Population 2000                                               81 non-null     float64
 2   Male Under 5 2000                                             81 non-null     float64
 3   Female Under 5 2000                                           81 non-null     float64
 4   Male 5 to 9 2000                                              81 non-null     float64
 5   Female 5 to 9 2000                                            81 non-null     float64
 6   Male 10 to 14 2000                                            81 non-null     float64
 7   Female 10 to 14 2000                                          81 non-null

In [16]:
data.to_csv('../../data/Prefix2000SF1.csv', index = False)